# Pi 机制：flow 符号、两层 attention 泄漏与时延

保留原标准库机制检查：不训练 Transformer，不使用真实 RFM，也不启动物理模拟。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)


In [ ]:
from __future__ import annotations
import json
import math
from pathlib import Path
import random


### 定义：`max_difference`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def max_difference(left: list[float], right: list[float]) -> float:
    if len(left) != len(right):
        raise ValueError("Compared vectors must have equal lengths.")
    return max(abs(a - b) for a, b in zip(left, right))


### 定义：`check_flow`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def check_flow(seed: int, steps: int) -> dict:
    rng = random.Random(seed)
    shape = [2, 4, 3]
    count = math.prod(shape)
    actions = [rng.uniform(-2.0, 2.0) for _ in range(count)]
    noise = [rng.gauss(0.0, 1.0) for _ in range(count)]
    target_forward = [a - e for a, e in zip(actions, noise)]
    target_reverse = [-u for u in target_forward]
    tau, h = 0.37, 1e-6

    def path(t: float) -> list[float]:
        return [(1 - t) * e + t * a for a, e in zip(actions, noise)]

    finite_difference = [(p - m) / (2 * h) for p, m in zip(path(tau + h), path(tau - h))]

    def integrate(velocity: list[float], dt: float) -> list[float]:
        # This is an endpoint-conditioned oracle, not an inference-time model.
        x = noise[:]
        for _ in range(steps):
            x = [value + dt * speed for value, speed in zip(x, velocity)]
        return x

    forward = integrate(target_forward, 1.0 / steps)
    reverse = integrate(target_reverse, -1.0 / steps)
    wrong_sign = integrate(target_reverse, 1.0 / steps)
    expected_wrong = [2 * e - a for a, e in zip(actions, noise)]
    derivative_error = max_difference(finite_difference, target_forward)
    forward_error = max_difference(forward, actions)
    reverse_error = max_difference(reverse, actions)
    convention_difference = max_difference(forward, reverse)
    wrong_error = max_difference(wrong_sign, actions)
    if derivative_error > 1e-8 or max(forward_error, reverse_error, convention_difference) > 1e-10:
        raise AssertionError("Consistent flow conventions did not agree with the path.")
    if wrong_error < 1e-3 or max_difference(wrong_sign, expected_wrong) > 1e-10:
        raise AssertionError("The deliberately inconsistent target/dt did not fail as expected.")
    return {
        "conceptual_tensor_shape": shape,
        "finite_difference_max_error": derivative_error,
        "forward_endpoint_max_error": forward_error,
        "reverse_endpoint_max_error": reverse_error,
        "convention_max_difference": convention_difference,
        "wrong_sign_endpoint_max_error": wrong_error,
        "limitation": "Known endpoints and constant oracle velocity; no learned generative policy.",
    }


### 定义：`build_mask`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def build_mask(leak_prefix: bool) -> list[list[bool]]:
    groups = ["P", "P", "D", "D", "C", "C", "PAD"]
    valid = [True, True, True, True, True, True, False]
    mask = []
    for i, query in enumerate(groups):
        row = []
        for j, key in enumerate(groups):
            allowed = (
                (query == "P" and (key == "P" or (leak_prefix and key == "D")))
                or (query == "D" and (key == "P" or (key == "D" and j <= i)))
                or (query == "C" and key in {"P", "C"})
            )
            row.append(valid[i] and valid[j] and allowed)
        mask.append(row)
    return mask


### 定义：`attention_layer`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def attention_layer(values: list[list[float]], mask: list[list[bool]]) -> list[list[float]]:
    """One scalar-head residual layer with Q=K=V for a transparent counterexample."""
    width = len(values[0])
    output = []
    for query, row in zip(values, mask):
        indices = [j for j, allowed in enumerate(row) if allowed]
        if not indices:
            output.append(query[:])
            continue
        scores = [sum(q * k for q, k in zip(query, values[j])) / math.sqrt(width) for j in indices]
        largest = max(scores)
        weights = [math.exp(score - largest) for score in scores]
        denominator = sum(weights)
        aggregate = [
            sum(weight * values[j][d] for weight, j in zip(weights, indices)) / denominator
            for d in range(width)
        ]
        output.append([q + a for q, a in zip(query, aggregate)])
    return output


### 定义：`action_difference`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def action_difference(left: list[list[float]], right: list[list[float]]) -> float:
    return max_difference([x for row in left[4:6] for x in row], [x for row in right[4:6] for x in row])


### 定义：`check_attention`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def check_attention() -> dict:
    initial = [[0.1, 0.2], [0.2, -0.1], [0.3, 0.4], [-0.2, 0.1], [0.1, -0.3], [0.4, 0.2], [9.0, 9.0]]
    perturbed = [row[:] for row in initial]
    perturbed[2] = [2.3, 1.4]  # Change clean FAST action targets only.
    correct_mask, leaky_mask = build_mask(False), build_mask(True)

    def twice(data: list[list[float]], mask: list[list[bool]]) -> list[list[float]]:
        return attention_layer(attention_layer(data, mask), mask)

    correct_delta = action_difference(twice(initial, correct_mask), twice(perturbed, correct_mask))
    leaky_one_delta = action_difference(attention_layer(initial, leaky_mask), attention_layer(perturbed, leaky_mask))
    leaky_two_delta = action_difference(twice(initial, leaky_mask), twice(perturbed, leaky_mask))
    padded = [row[:] for row in initial]
    padded[-1] = [-100.0, 100.0]
    pad_delta = action_difference(twice(initial, correct_mask), twice(padded, correct_mask))
    future_discrete = [row[:] for row in initial]
    future_discrete[3] = [3.0, -2.0]
    causal_delta = max_difference(twice(initial, correct_mask)[2], twice(future_discrete, correct_mask)[2])
    if max(correct_delta, leaky_one_delta, pad_delta, causal_delta) > 1e-12:
        raise AssertionError("A forbidden direct or indirect dependency leaked through the correct mask.")
    if leaky_two_delta < 1e-3:
        raise AssertionError("The deliberate D -> P -> C leakage was not detected.")
    return {
        "token_order": ["P0", "P1", "D0", "D1", "C0", "C1", "padding"],
        "correct_mask": [[int(v) for v in row] for row in correct_mask],
        "correct_two_layer_action_delta": correct_delta,
        "leaky_one_layer_action_delta": leaky_one_delta,
        "leaky_two_layer_action_delta": leaky_two_delta,
        "padding_perturbation_action_delta": pad_delta,
        "future_FAST_perturbation_past_FAST_delta": causal_delta,
        "limitation": "Hand-built identity Q/K/V attention; no trained Transformer or gradient routing test.",
    }


### 定义：`delay_case`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def delay_case(latency_ms: float, action_hz: int, execute_steps: int, horizon: int) -> dict:
    elapsed_intervals = math.ceil(latency_ms * action_hz / 1000.0)
    window_ms = 1000.0 * execute_steps / action_hz
    return {
        "latency_ms": latency_ms,
        "action_hz": action_hz,
        "execute_steps": execute_steps,
        "horizon": horizon,
        "execution_window_ms": window_ms,
        "async_budget_margin_ms_ignoring_other_overheads": window_ms - latency_ms,
        "elapsed_action_intervals_at_arrival": elapsed_intervals,
        "remaining_waypoints_after_elapsed_prefix": max(0, horizon - elapsed_intervals),
        "synchronous_cycle_ms": window_ms + latency_ms,
        "assumption": "Observation at time zero; inference overlaps execution only in async budget calculation; zero other overhead.",
    }


### 定义：`check_delay`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def check_delay() -> dict:
    cases = [delay_case(ms, 50, 10, 50) for ms in (0, 120, 260, 1200)]
    if cases[1]["elapsed_action_intervals_at_arrival"] != 6:
        raise AssertionError("120 ms at 50 Hz should span six intervals.")
    if cases[2]["async_budget_margin_ms_ignoring_other_overheads"] >= 0:
        raise AssertionError("260 ms inference should miss a 200 ms overlap budget.")
    if cases[3]["remaining_waypoints_after_elapsed_prefix"] != 0:
        raise AssertionError("An expired chunk must have no remaining waypoints.")
    return {
        "cases": cases,
        "limitation": "Clock arithmetic, not a scheduler, physical simulation, or recommendation to blindly discard contact actions.",
    }


## 2. 运行三个机制检查
Flow 部分使用已知端点的 oracle velocity；真正学习 velocity field 的实验见 ML06。


In [ ]:
STEPS = 10
results = {'seed': SEED, 'steps': STEPS, 'flow': check_flow(SEED, STEPS),
           'attention': check_attention(), 'delay': check_delay(),
           'scope': 'Mechanism checks only, not learned-model or robot validation.'}
print(json.dumps(results, indent=2))


## 3. 显示允许的 attention 依赖与异步预算


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mask_array = np.asarray(results['attention']['correct_mask'])
axes[0].imshow(mask_array, vmin=0, vmax=1)
labels = results['attention']['token_order']
axes[0].set_xticks(range(len(labels)), labels, rotation=45)
axes[0].set_yticks(range(len(labels)), labels)
axes[0].set(title='Allowed query-to-key dependencies', xlabel='key', ylabel='query')
cases = results['delay']['cases']
axes[1].bar([str(c['latency_ms']) for c in cases],
            [c['async_budget_margin_ms_ignoring_other_overheads'] for c in cases])
axes[1].axhline(0, color='black')
axes[1].set(title='200 ms overlap budget', xlabel='inference latency (ms)', ylabel='margin (ms)')
plt.tight_layout(); plt.show()


## 面试追问与实验记录
1. 为什么只翻 velocity target、不翻积分方向会错？
2. 为什么第一层没有泄漏、第二层却能泄漏？
3. 1 kHz 插值点、50 Hz policy 与 inference latency 是三个不同量。

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
